[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/Figures.ipynb)

Open this exact notebook in Colab, connect to a runtime, and run cells from the top using ▶. GitHub Preview displays the notebook without executing it.


# Reusable scientific figures

Or Shulrufer

This independent notebook defines reusable scientific and data science figures. It neither imports nor executes projects or dataset loaders. Set explicit input paths for measured-data figures and choose whether to render the illustrative examples.


## Failure reporting
Each operation reports what failed, the exception reason, and what to assess. Unexpected failures retain their traceback and stop execution. A missing project dataset is handled separately with a debug message and skipped dependent calculations.

In [ ]:
from contextlib import contextmanager
import subprocess

@contextmanager
def report_failure(stage, assessment):
    try:
        yield
    except Exception as error:
        print(f"FAILED: {stage}", flush=True)
        print(f"Reason: {type(error).__name__}: {error}", flush=True)
        if isinstance(error, subprocess.CalledProcessError):
            details = error.stderr or error.stdout or error.output
            if details:
                if isinstance(details, bytes):
                    details = details.decode("utf-8", errors="replace")
                print(f"Process output (last 4000 characters):\n{str(details)[-4000:]}", flush=True)
        print(f"Assess: {assessment}", flush=True)
        print("This operation failed. Correct the cause and rerun it before dependent cells.", flush=True)
        raise


## Automatic dependency setup

Run all cells without editing installation settings. Each setup cell checks installed packages and installs only missing packages using the declared versions. Existing versions are retained; differences from the declared versions are printed for reproducibility. Package presence alone does not establish compatibility. XeLaTeX is installed only when absent and installation permissions are available.


## Plotting libraries

Install this notebook’s numerical and plotting dependencies independently of project implementations.


In [ ]:
with report_failure('Plotting libraries', 'Check network access, package availability, Python compatibility, installation permissions and available disk space.'):
    import importlib.metadata
    import subprocess
    import sys

    def install_notebook_dependencies(packages):
        missing=[]
        for package in packages:
            name, version = package.split("==", 1)
            try: installed=importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError: installed=None
            if installed is None: missing.append(package)
            elif installed != version: print(f"{name}: keeping installed {installed}; declared version is {version}.")
        if missing:
            subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)

    install_notebook_dependencies(("numpy==2.3.5","scipy==1.17.0","matplotlib==3.10.8","scikit-learn==1.8.0"))


## Figure rendering libraries and fonts

The figure functions also require PDF image tools and XeLaTeX. Prepare those dependencies before defining and rendering the illustrations.


In [ ]:
with report_failure('Figure rendering libraries and fonts', 'Check network access, package availability, Python compatibility, installation permissions and available disk space.'):
    # Purpose: install the figure renderer's additional Python and TeX dependencies.
    install_notebook_dependencies(("PyMuPDF==1.26.6", "Pillow==12.3.0"))
    import shutil
    import os
    if not shutil.which("xelatex"):
        if shutil.which("apt-get") and hasattr(os, "geteuid") and os.geteuid() == 0:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "texlive-xetex", "texlive-latex-extra", "texlive-pictures", "texlive-fonts-recommended", "lmodern"], check=True)
        else:
            raise RuntimeError("Install XeLaTeX, texlive-latex-extra, texlive-pictures and Latin Modern fonts, then rerun this cell.")


## Figure inputs and verification

Establish the result locations, visual constants and verification guard used by the figure generators. Empirical plots require completed results for the declared corpus.


In [ ]:
with report_failure('Figure inputs and verification', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    # Purpose: generate all manuscript illustrations after the complete computational verification passes.
    from pathlib import Path
    import csv
    import hashlib
    import json
    import math
    import shutil
    import subprocess
    import tempfile
    from datetime import datetime, timezone
    import numpy as np
    from scipy import sparse
    import matplotlib
    from matplotlib.patches import Circle, FancyArrowPatch

    # Supply paths explicitly when rendering measured data; no project is loaded automatically.
    OUT = Path("figure_outputs").resolve()
    RESULTS_DIRECTORY = None
    FIGURES_APPROVED = True
    RENDER_FIGURES = False
    AUTHOR = "Or Shulrufer"
    BLUE, TEAL, GRAY, PALE = "#205F88", "#317D7B", "#5B6D7C", "#E1EDF5"
    ORANGE, EDGE = "#B36A24", "#335879"
    X = np.array([[1.,1.],[2.,1.],[1.,2.],[5.,4.],[6.,4.],[5.,5.]])
    values = {}

    def require_approval():
        if not FIGURES_APPROVED:
            raise RuntimeError("Figure rendering is deferred until the results and sources are approved.")

    def require_verified_results(folder):
        if folder is None:
            raise ValueError("Supply the directory of measured results explicitly.")
        record = json.loads((Path(folder) / "verification.json").read_text())
        if record.get("complete") is not True:
            raise ValueError("Measured results have not passed their completion checks.")
        return record

    def sha(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def utc():
        return datetime.now(timezone.utc).isoformat()

    def provenance(action, sources, author=AUTHOR):
        return {"created_at_utc": utc(), "created_by": author, "action": action,
                "sources": {str(Path(p).resolve()): sha(p) for p in sources}}


## Typography and plotting configuration

Configure the installed XeLaTeX renderer and Latin Modern fonts so all plots share the manuscript typography.


In [ ]:
with report_failure('Typography and plotting configuration', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def configure_plotting():
        if not shutil.which("xelatex"):
            raise RuntimeError("XeLaTeX is required to preserve the prescribed Latin Modern fonts.")
        import matplotlib
        matplotlib.use("pgf")
        import matplotlib.pyplot as plt
        matplotlib.rcParams.update({
            "pgf.texsystem": "xelatex", "pgf.rcfonts": False,
            "pgf.preamble": (
                r"\usepackage{fontspec}"
                r"\setmainfont{lmroman10-regular.otf}[BoldFont=lmroman10-bold.otf,"
                r"ItalicFont=lmroman10-italic.otf,BoldItalicFont=lmroman10-bolditalic.otf]"
                r"\usepackage{unicode-math}\setmathfont{latinmodern-math.otf}"
            ),
            "text.usetex": True, "font.family": "serif", "font.size": 10,
            "axes.labelsize": 10, "axes.titlesize": 10, "xtick.labelsize": 9,
            "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
            "axes.spines.top": False, "axes.spines.right": False,
            "axes.edgecolor": GRAY, "xtick.color": GRAY, "ytick.color": GRAY,
            "axes.linewidth": .8, "axes.axisbelow": True, "lines.linewidth": 1.3,
            "text.color": "black", "axes.labelcolor": "black", "figure.facecolor": "white",
            "axes.facecolor": "white", "savefig.facecolor": "white",
            "grid.color": PALE, "grid.linewidth": .5,
        })
        return plt

## Figure export

Define the shared export function used by the explanatory figures and record the resulting files.


In [ ]:
with report_failure('Figure export', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def save(fig, name):
        require_approval()
        import fitz
        OUT.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUT / (name + ".pdf"), bbox_inches="tight")
        with fitz.open(OUT / (name + ".pdf")) as document:
            document[0].get_pixmap(matrix=fitz.Matrix(300/72, 300/72), alpha=False).save(OUT / (name + ".png"))
        values[name] = {"scope": "Synthetic explanatory illustration", "pdf_sha256": sha(OUT / (name + ".pdf"))}
        plt.close(fig)


## Graph edges

Define the reusable connection primitive used to draw relationships in the graph illustrations.


In [ ]:
with report_failure('Graph edges', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def edge(ax,a,b,width=1.35,color=EDGE,style='-'):
        ax.plot([a[0],b[0]],[a[1],b[1]],lw=width,color=color,ls=style,zorder=1)

## Graph nodes

Define labeled nodes that combine with the edge primitive in the graph illustrations.


In [ ]:
with report_failure('Graph nodes', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def node(ax,xy,label,color=PALE,radius=.22,fs=9.5):
        ax.add_patch(Circle(xy,radius,facecolor=color,edgecolor='#25394c',lw=1.1,zorder=3))
        ax.text(*xy,label,ha='center',va='center',fontsize=fs,zorder=4)

## Graph coordinate frames

Define axis limits and presentation for the graph drawings so their geometry is consistent.


In [ ]:
with report_failure('Graph coordinate frames', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def graph_axis(ax,xlim=(-1.3,1.3),ylim=(-1.2,1.2)):
        ax.set(xlim=xlim,ylim=ylim);ax.set_aspect('equal');ax.axis('off')

## Illustrative point populations

Define the shared synthetic points used to explain distances and groups. These points are explanatory examples rather than corpus findings.


In [ ]:
with report_failure('Illustrative point populations', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def points(ax,groups=False):
        if groups:
            ax.scatter(X[:3,0],X[:3,1],c='#303030',marker='o',s=25,label='Group 1')
            ax.scatter(X[3:,0],X[3:,1],c='#888888',marker='s',s=25,label='Group 2')
        else:ax.scatter(X[:,0],X[:,1],c='#303030',s=25)
        for i,(x,y) in enumerate(X):
            offsets=[(-11,-14),(-2,-14),(-12,5),(-13,-14),(4,-13),(-12,6)]
            ax.annotate('$x_'+str(i+1)+'$',(x,y),xytext=offsets[i],textcoords='offset points',fontsize=9)
        ax.set(xlim=(0,7),ylim=(0,6),xlabel='Feature 1',ylabel='Feature 2')
        ax.set_xticks(range(8));ax.set_yticks(range(7));ax.grid(alpha=.15)
        ax.set_aspect('equal')

## Bridges and graph connectivity

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Bridges and graph connectivity', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_bridges():
        require_approval()
        fig,axs=plt.subplots(1,2,figsize=(6.5,3.05),gridspec_kw={'width_ratios':[1,1.15]})
        graph_axis(axs[0],(-1.0,1.6),(-1.15,1.25))
        pos={'A':(-.05,0),'B':(-.55,-.83),'C':(-.55,.83),'D':(1.1,0)}
        bridges=[('A','B',.22),('A','B',-.22),('A','C',.22),('A','C',-.22),('A','D',0),('B','D',0),('C','D',0)]
        for u,v,rad in bridges:
            axs[0].add_patch(FancyArrowPatch(pos[u],pos[v],arrowstyle='-',linewidth=1.4,
                color=EDGE,connectionstyle=f'arc3,rad={rad}',shrinkA=14,shrinkB=14))
        for n,xy in pos.items():node(axs[0],xy,n,radius=.19,fs=11)
        axs[0].set_title('Seven bridge incidences')
        M=np.array([[0,2,2,1],[2,0,0,1],[2,0,0,1],[1,1,1,0]])
        assert M.sum(1).tolist()==[5,3,3,3] and M.sum()==14
        axs[1].axis('off')
        axs[1].set_title('Parallel-edge counts')
        tab=axs[1].table(cellText=M.astype(str),rowLabels=list('ABCD'),colLabels=list('ABCD'),cellLoc='center',bbox=[.12,.30,.76,.60])
        tab.auto_set_font_size(False)
        tab.set_fontsize(9.5)
        axs[1].text(.02,.17,'Degree sequence: 5, 3, 3, 3',transform=axs[1].transAxes)
        axs[1].text(.02,.04,'Four odd vertices: no Eulerian trail',transform=axs[1].transAxes,fontsize=9)
        fig.subplots_adjust(left=.025,right=.98,bottom=.09,top=.87,wspace=.20)
        save(fig,'Work_bridges')

## Graph topology

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Graph topology', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_topology():
        require_approval()
        fig,axs=plt.subplots(1,3,figsize=(6.5,2.75),gridspec_kw={'width_ratios':[1,1,1.10]})
        cycle=[('a','b'),('b','c'),('c','d'),('d','a')]
        positions=[{'a':(-.8,.7),'b':(.8,.7),'c':(.8,-.7),'d':(-.8,-.7)},
                   {'a':(-.8,.7),'b':(.8,-.15),'c':(-.7,-.7),'d':(.14,.5)}]
        for ax,pos,title in zip(axs[:2],positions,['Drawing 1','Drawing 2']):
            graph_axis(ax,(-1.2,1.2),(-1.05,1.08))
            for u,v in cycle:edge(ax,pos[u],pos[v])
            for n,xy in pos.items():node(ax,xy,n,radius=.20)
            ax.set_title(title)
        A=np.array([[0,1,0,1],[1,0,1,0],[0,1,0,1],[1,0,1,0]])
        assert np.allclose(np.linalg.eigvalsh(np.diag(A.sum(1))-A),[0,2,2,4])
        axs[2].axis('off')
        axs[2].set_title('Adjacency and spectrum')
        tab=axs[2].table(cellText=A.astype(str),rowLabels=list('abcd'),colLabels=list('abcd'),cellLoc='center',bbox=[.12,.35,.78,.56])
        tab.auto_set_font_size(False)
        tab.set_fontsize(9)
        axs[2].text(.08,.19,'Laplacian eigenvalues',transform=axs[2].transAxes,fontsize=9)
        axs[2].text(.08,.08,'0, 2, 2, 4',transform=axs[2].transAxes)
        fig.subplots_adjust(left=.02,right=.98,bottom=.08,top=.85,wspace=.20)
        save(fig,'Work_topology')

## Semantic relationships

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Semantic relationships', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_semantic():
        require_approval()
        fig,axs=plt.subplots(1,2,figsize=(6.5,3.15))
        for ax in axs:graph_axis(ax,(-1.15,2.0),(-1.18,1.18))
        left={'bank':(-.7,0),'sense 1':(.15,.65),'sense 2':(.15,-.65),'finance':(1.3,.65),'river\nmargin':(1.3,-.65)}
        for u,v in [('bank','sense 1'),('bank','sense 2'),('sense 1','finance'),('sense 2','river\nmargin')]:edge(axs[0],left[u],left[v])
        for n,xy in left.items():node(axs[0],xy,n,'#EFE8D9' if n=='bank' else PALE,radius=.31,fs=9)
        axs[0].set_title('Interpreted sense relations')
        right={'bank':(-.65,0),'money':(.55,.85),'loan':(1.5,.5),'river':(.55,-.85),'shore':(1.5,-.5)}
        for u,v,w in [('bank','money',1.6),('bank','loan',1.9),('bank','river',1.5),('bank','shore',1.1),('money','loan',1.1),('river','shore',1.1)]:edge(axs[1],right[u],right[v],w)
        for n,xy in right.items():node(axs[1],xy,n,'#DFE8DA' if n=='bank' else 'white',radius=.31,fs=9)
        axs[1].set_title('Corpus co-occurrence graph')
        fig.subplots_adjust(left=.02,right=.98,bottom=.05,top=.88,wspace=.15)
        save(fig,'Work_semantic')

## Coordinate representations

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Coordinate representations', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_coordinates():
        require_approval()
        fig,axs=plt.subplots(1,2,figsize=(6.5,3.65),gridspec_kw={'width_ratios':[1,1.05]})
        axs[0].axis('off')
        axs[0].set_title('Six observations, two features')
        tab=axs[0].table(cellText=[[f'$x_{i+1}$',str(int(x)),str(int(y))] for i,(x,y) in enumerate(X)],colLabels=['Observation','Feature 1','Feature 2'],cellLoc='center',bbox=[.02,.19,.96,.72])
        tab.auto_set_font_size(False)
        tab.set_fontsize(9.5)
        for c in tab.get_celld().values():c.set_linewidth(0)
        axs[0].text(.02,.10,'Table and points contain the same values.',transform=axs[0].transAxes,fontsize=9)
        axs[0].text(.02,.035,'Synthetic data, no assumed group labels.',transform=axs[0].transAxes,fontsize=9)
        points(axs[1])
        fig.subplots_adjust(left=.02,right=.98,bottom=.15,top=.88,wspace=.24)
        save(fig,'Work_coordinates')

## Euclidean distances

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Euclidean distances', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_distance():
        require_approval()
        fig,ax=plt.subplots(figsize=(4.6,4.05))
        ax.set(xlim=(0,7),ylim=(0,6),xlabel='Feature 1',ylabel='Feature 2')
        ax.set_aspect('equal')
        ax.set_title('Euclidean distance and orthogonal axes')
        ax.grid(alpha=.15)
        ax.plot([1,5],[1,4],color='#222222',lw=1.4)
        ax.scatter([1,5],[1,4],c='#222222',s=25,zorder=3)
        ax.plot([1,5],[1,1],ls='--',color='#555555',lw=1)
        ax.plot([5,5],[1,4],ls=':',color='#555555',lw=1.3)
        ax.plot([4.80,4.80,5],[1,1.20,1.20],color=BLUE,lw=.8)
        ax.text(.85,.58,'$x_1=(1,1)$',fontsize=9)
        ax.text(5.07,4.12,'$x_4=(5,4)$',fontsize=9)
        ax.text(2.50,.66,r'$\Delta f_1=4$',fontsize=9.5)
        ax.text(5.12,2.38,r'$\Delta f_2=3$',fontsize=9.5)
        ax.text(2.35,2.85,'$d(x_1,x_4)=5$',rotation=36.87,fontsize=9.5)
        fig.subplots_adjust(left=.13,right=.98,bottom=.14,top=.89)
        save(fig,'Work_distance')

## Within-group and between-group variance

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Within-group and between-group variance', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_variance():
        require_approval()
        fig,ax=plt.subplots(figsize=(4.6,4.1))
        points(ax,True)
        mu=np.array([X[:3].mean(0),X[3:].mean(0)])
        grand=X.mean(0)
        for i,x in enumerate(X):edge(ax,x,mu[i//3],.9,'#777777',':')
        for g in mu:edge(ax,g,grand,1.3,'#303030','--')
        ax.scatter(mu[:,0],mu[:,1],marker='X',s=48,color='#222222',zorder=4,label='Group means')
        ax.scatter(*grand,marker='*',s=85,color='#222222',zorder=5,label='Overall mean')
        for i,(x,y) in enumerate(mu):ax.annotate(r'$\mu_'+str(i+1)+'$',(x,y),xytext=(-21,4) if i==0 else (5,6),textcoords='offset points',fontsize=9.5)
        ax.annotate(r'$\mu$',grand,xytext=(7,-10),textcoords='offset points',fontsize=9.5)
        ax.plot([],[],':',color='#777777',lw=.9,label='Within group')
        ax.plot([],[],'--',color='#303030',lw=1.3,label='Between groups')
        ax.legend(loc='upper left',ncol=2,columnspacing=.8,handlelength=1.8,frameon=True,framealpha=1,fontsize=8.5)
        ax.set_title('Within and between group deviations')
        assert np.allclose([sum(((X[g]-X[g].mean(0))**2).sum() for g in [slice(0,3),slice(3,6)]),sum(3*np.sum((v-grand)**2) for v in mu),np.sum((X-grand)**2)],[8/3,75/2,241/6])
        fig.subplots_adjust(left=.13,right=.98,bottom=.14,top=.89)
        save(fig,'Work_variance')

## Tree structure

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Tree structure', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_tree():
        require_approval()
        fig,ax=plt.subplots(figsize=(4.6,4.1))
        points(ax,True)
        for i,j in [(0,1),(0,2),(1,3),(3,4),(3,5)]:edge(ax,X[i],X[j],1.35,'#888888' if (i,j)==(1,3) else '#333333','--' if (i,j)==(1,3) else '-')
        ax.annotate('Remove this edge for two components',xy=(3.5,2.5),xytext=(.4,4.65),arrowprops={'arrowstyle':'->','lw':.8},fontsize=9)
        ax.plot([],[],'--',color='#888888',label='Removed edge')
        ax.legend(loc='upper right',fontsize=8.5,framealpha=1)
        ax.set_title('Minimum spanning tree and one edge removal')
        fig.subplots_adjust(left=.13,right=.98,bottom=.14,top=.89)
        save(fig,'Work_tree')

## Coordinate projections

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Coordinate projections', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_projections():
        require_approval()
        B=np.array([[1,1,0,0,0],[1,1,0,0,0],[1,0,1,0,0],[1,0,1,1,0],[0,1,0,0,1],[0,0,1,1,0]])
        R=B@B.T
        K=B.T@B
        fig,axs=plt.subplots(1,3,figsize=(6.5,3.25),gridspec_kw={'width_ratios':[1.06,1,1]})
        ax=axs[0]
        ax.imshow(B,cmap='Blues',vmin=0,vmax=1,interpolation='nearest',aspect='auto')
        ax.set_xticks(range(5),['bank','river','loan','credit','flow'],rotation=45,ha='right',fontsize=8.5)
        ax.set_yticks(range(6),[f'$s_{i}$' for i in range(1,7)],fontsize=9)
        for i in range(6):
            for j in range(5):ax.text(j,i,str(B[i,j]),ha='center',va='center',color='white' if B[i,j] else '#303030',fontsize=9)
        ax.set_title('Context-word incidence')
        ax=axs[1]
        graph_axis(ax,(-1.35,1.35),(-1.3,1.3))
        pos={i:(np.cos(2*np.pi*i/6),np.sin(2*np.pi*i/6)) for i in range(6)}
        for i in range(6):
            for j in range(i+1,6):
                if R[i,j]>0:edge(ax,pos[i],pos[j],.5+.5*R[i,j])
        for i,xy in pos.items():node(ax,xy,f'$s_{i+1}$',radius=.23,fs=9)
        ax.set_title('Context projection')
        ax=axs[2]
        graph_axis(ax,(-1.40,1.40),(-1.3,1.3))
        pos={i:(np.cos(2*np.pi*i/5),np.sin(2*np.pi*i/5)) for i in range(5)}
        names=['bank','river','loan','credit','flow']
        for i in range(5):
            for j in range(i+1,5):
                if K[i,j]>0:edge(ax,pos[i],pos[j],.5+.5*K[i,j])
        for i,xy in pos.items():node(ax,xy,names[i],color='#EFE8D9',radius=.29,fs=8.5)
        ax.set_title('Vocabulary projection')
        fig.subplots_adjust(left=.055,right=.98,bottom=.22,top=.85,wspace=.25)
        save(fig,'Work_projections')

## Structural roles

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Structural roles', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_roles():
        require_approval()
        fig,axs=plt.subplots(1,2,figsize=(6.5,3.25),gridspec_kw={'width_ratios':[1.1,1]})
        xx=np.arange(1,5)
        a=np.array([.72,.63,.68,.70])
        b=np.array([.05,.10,.06,.13])
        assert np.allclose([a.mean(),b.mean()],[.6825,.085])
        for yy,cl,m,label in [(a,BLUE,'o','Group 1'),(b,ORANGE,'s','Group 2')]:
            axs[0].scatter(xx,yy,s=26,color=cl,marker=m,label=label,zorder=3)
            axs[0].hlines(yy.mean(),1,4,color=cl,ls='--',lw=1.2)
        axs[0].set(xlim=(.6,4.4),ylim=(0,.87),xlabel='Observation within its group',ylabel='TF-IDF feature weight')
        axs[0].set_xticks(xx)
        axs[0].legend(frameon=False,fontsize=9,loc='center right')
        axs[0].grid()
        ax=axs[1]
        graph_axis(ax,(-1.5,1.5),(-1.15,1.35))
        pos={'a':(-1,.67),'b':(-1,-.55),'t':(0,.05),'c':(1,.67),'d':(1,-.55)}
        for u,v in [('a','b'),('a','t'),('b','t'),('t','c'),('t','d'),('c','d')]:edge(ax,pos[u],pos[v],1.35)
        for n,xy in pos.items():node(ax,xy,n,'#EFE8D9' if n in ('a','b','t') else PALE,radius=.23)
        ax.text(-1,1.04,'Community 1',ha='center',fontsize=9)
        ax.text(1,1.04,'Community 2',ha='center',fontsize=9)
        ax.text(0,-.98,'Term $t$ belongs to community 1',ha='center',fontsize=9)
        fig.subplots_adjust(left=.10,right=.98,bottom=.20,top=.96,wspace=.22)
        save(fig,'Work_roles')

## Word prevalence

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Word prevalence', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_prevalence():
        require_approval()
        fig,axs=plt.subplots(1,2,figsize=(6.5,3.05),gridspec_kw={'width_ratios':[1.08,1]})
        x=np.arange(1,13)
        a=np.where(x<=2,1.,0.)
        b=np.where(x<=8,1.,0.)
        axs[0].scatter(x,a,s=24,label='A: 2 of 12',color=BLUE,zorder=3)
        axs[0].scatter(x,b,s=26,label='B: 8 of 12',marker='s',color=ORANGE,zorder=2,alpha=.85)
        axs[0].set(xlim=(.5,12.5),ylim=(-.12,1.2),xlabel='Observation index',ylabel='Feature weight')
        axs[0].set_xticks(x)
        axs[0].legend(frameon=False,fontsize=9,loc='center right')
        axs[0].grid()
        pp=np.linspace(.045,1.,300)
        axs[1].plot(pp,np.sqrt((1-pp)/pp),color=BLUE,lw=1.4)
        for n,cl in [(2,BLUE),(8,ORANGE)]:
            pr=n/12;axs[1].scatter([pr],[np.sqrt((1-pr)/pr)],s=28,color=cl,zorder=4)
        axs[1].set(xlim=(0,1.02),ylim=(0,4.6),xlabel='Positive-weight prevalence',ylabel='Coefficient of variation')
        axs[1].grid()
        fig.subplots_adjust(left=.09,right=.98,bottom=.21,top=.97,wspace=.38)
        assert np.allclose([a.std()/a.mean(),b.std()/b.mean()],[np.sqrt(5),np.sqrt(.5)])
        save(fig,'Work_prevalence')

## Variance in three dimensions

Define the three-dimensional explanatory figure that separates point positions, group centers and dispersion.


In [ ]:
with report_failure('Variance in three dimensions', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_variance_three_dimensions():
        require_approval()
        from matplotlib.lines import Line2D
        from sklearn.cluster import KMeans
        from sklearn.metrics import calinski_harabasz_score
        offsets=np.array([[-.4,-.2,-.1],[.35,-.15,.25],[.15,.4,-.3],[-.1,-.05,.15]])
        seeds=np.array([[4.8,1.1,1.4],[1.3,4.8,1.8],[1.7,1.4,4.8]])
        X=np.concatenate([center+offsets for center in seeds])
        X=X/np.linalg.norm(X,axis=1,keepdims=True)
        assert np.allclose(np.linalg.norm(X,axis=1),1)
        model=KMeans(n_clusters=3,random_state=0,n_init=20,algorithm='lloyd',tol=0).fit(X)
        labels=model.labels_
        centers=np.stack([X[labels==j].mean(0) for j in range(3)])
        overall=X.mean(0)
        sizes=np.bincount(labels,minlength=3)
        within=float(((X-centers[labels])**2).sum())
        between=float(sum(sizes[j]*np.sum((centers[j]-overall)**2) for j in range(3)))
        total=float(((X-overall)**2).sum())
        ratio=(between/2)/(within/9)
        assert np.isclose(total,within+between)
        assert np.isclose(ratio,calinski_harabasz_score(X,labels))

        plt.rcParams.update({'font.family':'serif','font.size':10,'pdf.fonttype':42,
         'axes.labelsize':12,'axes.titlesize':12,'pgf.texsystem':'xelatex','pgf.rcfonts':False,
         'pgf.preamble':r'\usepackage{fontspec}\setmainfont{lmroman10-regular.otf}[BoldFont=lmroman10-bold.otf,ItalicFont=lmroman10-italic.otf,BoldItalicFont=lmroman10-bolditalic.otf]\usepackage{unicode-math}\setmathfont{latinmodern-math.otf}'})
        fig=plt.figure(figsize=(6.5,5.4),facecolor='white')
        ax=fig.add_axes([.03,.21,.60,.68],projection='3d')
        colors=['#0072B2','#D55E00','#009E73']
        for g,color in enumerate(colors):
         rows=X[labels==g]
         for row in rows:
          ax.plot(*np.vstack([row,centers[g]]).T,color=color,linewidth=1.35,alpha=.85)
         ax.scatter(*rows.T,s=28,color=color,depthshade=False,edgecolors='white',linewidths=.5)
         ax.scatter(*centers[g],s=95,marker='D',facecolors='white',edgecolors=color,linewidths=1.8,depthshade=False)
         line=np.vstack([centers[g],overall])
         ax.plot(*line.T,color='#333333',linestyle='--',linewidth=1.6)
        ax.scatter(*overall,s=190,marker='*',color='#111111',depthshade=False)
        ax.set_xlabel('Feature 1',labelpad=6)
        ax.set_ylabel('Feature 2',labelpad=6)
        ax.set_zlabel('Feature 3',labelpad=4)
        ax.view_init(elev=24,azim=35)
        ax.set_box_aspect((1,1,1))
        ax.set_proj_type('ortho')
        ax.set(xlim=(0,1.05),ylim=(0,1.05),zlim=(0,1.05))
        for axis in [ax.xaxis,ax.yaxis,ax.zaxis]:
         axis.set_pane_color((.97,.97,.97,1))
         axis._axinfo['grid'].update({'color':(.85,.85,.85,1),'linewidth':.5})
        ax.tick_params(labelsize=9,pad=0)
        fig.suptitle('Unit length passages and centroid variation',x=.5,y=.965,fontweight='bold',fontsize=12.5)
        fig.text(.665,.84,'WITHIN CLUSTERS',fontsize=10,fontweight='bold',color='#0072B2')
        fig.text(.665,.805,'Passage to its own centroid\nSolid colored segments\nSmaller spread tightens groups',fontsize=9.5,linespacing=1.4,va='top')
        fig.text(.665,.665,'BETWEEN CLUSTERS',fontsize=10,fontweight='bold')
        fig.text(.665,.63,'Cluster centroid to corpus centroid\nDashed dark segments\nLarger separation raises the ratio',fontsize=9.5,linespacing=1.4,va='top')
        fig.text(.665,.49,'ADD SQUARED DISTANCES',fontsize=9.5,fontweight='bold')
        fig.text(.665,.455,'Between group contributions are\nweighted by cluster size.',fontsize=9.5,linespacing=1.4,va='top')
        fig.text(.665,.35,'After the count adjustments',fontsize=9.5)
        fig.text(.665,.315,f'Between: {between/2:.6f}',fontsize=9.5)
        fig.text(.665,.28,f'Within: {within/9:.6f}',fontsize=9.5)
        fig.text(.665,.23,f'Caliński–Harabasz: {ratio:.3f}',fontsize=10,fontweight='bold')
        legend=[Line2D([0],[0],marker='o',color='none',markerfacecolor='#777777',markeredgecolor='white',markersize=6,label='Passage'),
         Line2D([0],[0],marker='D',color='none',markerfacecolor='white',markeredgecolor='#333333',markersize=7,label='Cluster centroid'),
         Line2D([0],[0],marker='*',color='none',markerfacecolor='#111111',markersize=11,label='Full corpus centroid')]
        fig.legend(handles=legend,loc='lower left',bbox_to_anchor=(.06,.12),ncol=3,frameon=False,fontsize=9.5,columnspacing=1.0,handletextpad=.5)
        fig.text(.06,.075,'Synthetic geometry: twelve unit length passages; three illustrative groups of four.',fontsize=9,color='#444444')
        fig.text(.06,.045,'Projected view; all distances are calculated in three dimensions.',fontsize=9,color='#444444')

        save(fig, "Work_Cluster_Variance_3D")


## Typeset diagram rendering

Define the TeX preamble and PDF-to-image renderer used by the method diagrams. The supplied diagram bodies are rendered without generating manuscript prose.


In [ ]:
with report_failure('Typeset diagram rendering', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    TEX_PREAMBLE = '\\documentclass[10pt,letterpaper]{article}\n\\usepackage[margin=1in]{geometry}\n\\usepackage{fontspec}\n\\setmainfont{lmroman10-regular.otf}[BoldFont=lmroman10-bold.otf,ItalicFont=lmroman10-italic.otf,BoldItalicFont=lmroman10-bolditalic.otf]\n\\usepackage{unicode-math}\n\\setmathfont{latinmodern-math.otf}\n\\usepackage{amsmath}\n\\usepackage{microtype,float}\n\\usepackage{array,tabularx,longtable,booktabs}\n\\usepackage{needspace}\n\\usepackage{placeins}\n\\usepackage{titlesec}\n\\usepackage{caption}\n\\usepackage{adjustbox}\n\\usepackage{tikz,pgfplots}\n\\usetikzlibrary{arrows.meta,positioning,fit,shapes.geometric,shapes.misc}\n\\pgfplotsset{compat=1.18}\n\\usepackage[hidelinks,breaklinks]{hyperref}\n\\hypersetup{pdftitle={From Linguistic Context to Semantic Structure},pdfauthor={Or Shulrufer}}\n\\urlstyle{same}\n\\setlength{\\parindent}{0pt}\n\\setlength{\\parskip}{6pt}\n\\setlength{\\emergencystretch}{1.5em}\n\\hyphenpenalty=500\n\\exhyphenpenalty=50\n\\setcounter{secnumdepth}{0}\n\\pagestyle{plain}\n\\titleformat{\\section}{\\fontsize{12}{14}\\selectfont\\bfseries}{}{0pt}{}\n\\titlespacing*{\\section}{0pt}{18pt}{9pt}\n\\titleformat{\\subsection}{\\fontsize{10}{12}\\selectfont\\bfseries}{}{0pt}{}\n\\titlespacing*{\\subsection}{0pt}{15pt}{7pt}\n\\captionsetup{font=small,labelfont=bf,justification=justified,singlelinecheck=false}\n\\setlength{\\textfloatsep}{14pt}\n\\setlength{\\intextsep}{12pt}\n\\setlength{\\abovedisplayskip}{6pt}\n\\setlength{\\belowdisplayskip}{4pt}\n\\newcommand{\\notlabel}{\\noindent\\textbf{Notation}\\par\\nobreak\\vspace{4pt}}\n\\hypersetup{pdfsubject={Corpus measurement and occurrence-level contextual learning}}'

    def render_diagram(name, body):
        require_approval()
        if not shutil.which("xelatex"):
            raise RuntimeError("Install XeLaTeX and Latin Modern fonts before rendering.")
        OUT.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix="figure-build-") as folder:
            folder = Path(folder)
            path = folder / (name + ".tex")
            path.write_text(TEX_PREAMBLE + "\n" + r"\begin{document}\pagestyle{empty}" + "\n" + body + "\n" + r"\end{document}")
            subprocess.run(["xelatex", "-interaction=nonstopmode", "-halt-on-error", path.name], cwd=folder, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            import fitz
            from PIL import Image, ImageChops
            with fitz.open(path.with_suffix(".pdf")) as document:
                assert document.page_count == 1
                page = document[0]
                pix = page.get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
                image = Image.frombytes("RGB", [pix.width,pix.height], pix.samples)
                bounds = ImageChops.difference(image, Image.new("RGB",image.size,"white")).getbbox()
                if bounds:
                    rect = fitz.Rect(max(0,bounds[0]/2-5),max(0,bounds[1]/2-5),min(page.rect.width,bounds[2]/2+5),min(page.rect.height,bounds[3]/2+5))
                    page.set_cropbox(rect)
                document.save(OUT / (name + ".pdf"))
                page.get_pixmap(matrix=fitz.Matrix(300/72,300/72),alpha=False).save(OUT / (name + ".png"))
        values[name] = {"scope":"Explanatory method diagram", "pdf_sha256":sha(OUT / (name + ".pdf"))}


## Cluster selection and distance aggregation

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Cluster selection and distance aggregation', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_ksearch():
        require_approval()
        body = '\n\\centering\n\\begin{adjustbox}{max width=\\linewidth}\n\\begingroup%\n\\hyphenpenalty=10000\\exhyphenpenalty=10000%\n\\definecolor{WorkKSBlue}{HTML}{205F8A}%\n\\definecolor{WorkKSTeal}{HTML}{267F88}%\n\\definecolor{WorkKSInk}{HTML}{26343E}%\n\\definecolor{WorkKSRule}{HTML}{D5E2EA}%\n\\begin{tikzpicture}[x=1cm,y=1cm,>=Stealth,\n  font=\\fontsize{9}{11}\\selectfont,text=WorkKSInk,\n  WorkKSStage/.style={circle,draw=WorkKSBlue,fill=WorkKSBlue,text=white,\n    minimum size=6.5mm,inner sep=0pt,font=\\fontsize{9.5}{11}\\selectfont\\bfseries},\n  heading/.style={font=\\fontsize{10}{12}\\selectfont\\bfseries,text=WorkKSBlue},\n  note/.style={align=left,anchor=north west,inner sep=0pt},\n  flow/.style={->,draw=WorkKSBlue,line width=.8pt}]\n\\node[heading,anchor=west] at (0,1.55){Choose the contextual reference};\n\\node[anchor=west] at (0,1.02){One fixed, row normalized TF IDF matrix $X$ in the original vocabulary space};\n\\node[anchor=west] at (0,.52){Declared candidates $k\\in K\\subseteq\\{2,\\ldots,n-1\\}$, and the complete reference is used each time.};\n\\node[WorkKSStage] (fit) at (.35,-.35){1};\n\\node[WorkKSStage] (score) at (5.65,-.35){2};\n\\node[WorkKSStage,draw=WorkKSTeal,fill=WorkKSTeal] (keep) at (10.65,-.35){3};\n\\draw[flow] (fit.east)--(score.west);\n\\draw[flow] (score.east)--(keep.west);\n\\node[heading,anchor=west] at (0,-.99){Fit the next $k$};\n\\node[heading,anchor=west] at (5.28,-.99){Measure the partition};\n\\node[heading,anchor=west,text=WorkKSTeal] at (10.3,-.99){Keep the best};\n\\node[note,text width=4.25cm] at (0,-1.47){Fit Euclidean k means to $X$.\\\\Assign each passage, then\\\\update arithmetic centers.};\n\\node[note,text width=4.2cm] at (5.28,-1.40){\\strut$\\displaystyle\\mathrm{CH}(k)=\\frac{\\mathrm{SSB}/(k-1)}{\\mathrm{SSW}/(n-k)}$};\n\\node[note,text width=4.3cm] at (10.3,-1.47){Retain the greatest finite CH.\\\\Store its $k$ and fitted centers.\\\\Exact ties retain smaller $k$.};\n\\node[note,text width=4.6cm] at (5.28,-2.38){SSB: weighted center separation.\\\\SSW: squared passage residuals.};\n\\draw[->,WorkKSBlue,line width=.75pt,dashed]\n  (10.65,-2.83)--(10.65,-3.25)--(-.4,-3.25)--(-.4,-.35)--(fit.west);\n\\node[fill=white,inner xsep=6pt,inner ysep=1pt,text=WorkKSBlue] at (3.2,-3.25){Repeat for every candidate};\n\\node[anchor=west,font=\\fontsize{8.5}{10.5}\\selectfont] at (0,-3.78)\n  {Admissible: $1<k<n$, nonempty groups, $\\mathrm{SSW}>0$ and finite CH.};\n\\draw[WorkKSRule,line width=.7pt] (0,-4.13)--(15.05,-4.13);\n\\draw[->,WorkKSTeal,line width=.85pt] (12.9,-2.83)--(12.9,-4.63);\n\\node[heading,anchor=west,text=WorkKSTeal] at (0,-4.53){After the candidate search};\n\\node[note,text width=4.3cm] at (0,-5.05){\\textbf{One full centroid}\\\\Select one centroid for each\\\\complete passage vector.};\n\\node[note,text width=4.6cm] at (5.2,-5.05){\\textbf{Match the word axis}\\\\$R[i,t]=|X[i,t]-\\mu[a[i],t]|$\\\\Retain absolute distances.};\n\\node[note,text width=4.3cm] at (10.7,-5.05){\\textbf{Aggregate all passages}\\\\$\\mathrm{CV}[t]=\\sigma[t]/m[t]$\\\\Population SD / mean.};\n\\draw[flow,draw=WorkKSTeal] (4.35,-5.56)--(4.95,-5.56);\n\\draw[flow,draw=WorkKSTeal] (9.82,-5.56)--(10.47,-5.56);\n\\node[anchor=west,font=\\fontsize{8.5}{10.5}\\selectfont] at (0,-6.47)\n  {No admissible count: one center, CH unavailable. Zero distance mean: CV unavailable.};\n\\end{tikzpicture}%\n\\endgroup%\n\n\\end{adjustbox}'
        render_diagram("Work_ksearch", body)


## Passage–word records

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Passage–word records', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_longformat():
        require_approval()
        body = '\n\\centering\n\\begin{adjustbox}{max width=\\linewidth}\n\\begingroup%\n\\hyphenpenalty=10000\\exhyphenpenalty=10000%\n\\definecolor{WorkLFBlue}{HTML}{205F8A}%\n\\definecolor{WorkLFTeal}{HTML}{267F88}%\n\\definecolor{WorkLFInk}{HTML}{26343E}%\n\\definecolor{WorkLFRule}{HTML}{D5E2EA}%\n\\begin{tikzpicture}[x=1cm,y=1cm,>=Stealth,\n  font=\\fontsize{9}{10.5}\\selectfont,text=WorkLFInk,\n  title/.style={font=\\fontsize{10}{12}\\selectfont\\bfseries,text=WorkLFBlue,anchor=west},\n  word/.style={circle,draw=WorkLFBlue,fill=WorkLFBlue!8,line width=.8pt,\n    minimum size=10mm,inner sep=2pt},\n  cluster/.style={rectangle,rounded corners=1pt,draw=WorkLFTeal,fill=WorkLFTeal!8,\n    line width=.8pt,minimum width=13mm,minimum height=8mm,inner sep=3pt}]\n\\node[title] at (0,9.3){(a) Select one full centroid};\n\\begin{scope}[shift={(.7,6.6)},x=2.1cm,y=2.1cm]\n  \\foreach \\v in {0,.5,1}{\n    \\draw[WorkLFRule,line width=.25pt] (\\v,0)--(\\v,1);\n    \\draw[WorkLFRule,line width=.25pt] (0,\\v)--(1,\\v);\n  }\n  \\draw[->,WorkLFInk,line width=.5pt] (0,0)--(1.17,0);\n  \\draw[->,WorkLFInk,line width=.5pt] (0,0)--(0,1.13);\n  \\node[anchor=east,font=\\fontsize{8.5}{10}\\selectfont] at (-.045,0){0};\n  \\node[anchor=north,font=\\fontsize{8.5}{10}\\selectfont] at (1,-.025){1};\n  \\node[anchor=east,font=\\fontsize{8.5}{10}\\selectfont] at (-.045,1){1};\n  \\node[anchor=west] at (1.17,0){$a$};\n  \\node[anchor=east] at (-.08,.6){$b$};\n  \\draw[WorkLFBlue!55,line width=.7pt] (.948683,.316228)--(.982894,.105409);\n  \\draw[WorkLFBlue,dashed,line width=.6pt]\n    (.948683,.316228)--(.982894,.316228)--(.982894,.105409);\n  \\fill[WorkLFBlue] (1,0) circle[radius=2pt];\n  \\fill[WorkLFBlue] (.948683,.316228) circle[radius=2pt];\n  \\draw[WorkLFTeal,fill=white,line width=.9pt] (-.035,.965) rectangle ++(.07,.07);\n  \\draw[WorkLFTeal,fill=white,line width=.9pt] (.281228,.913683) rectangle ++(.07,.07);\n  \\node[anchor=west,text=WorkLFBlue] at (1.025,.085){1,2};\n  \\node[anchor=east,text=WorkLFBlue] at (.91,.38){3};\n  \\node[anchor=west,text=WorkLFTeal] at (.045,1.065){4,5};\n  \\node[anchor=west,text=WorkLFTeal] at (.36,.91){6};\n  \\node[diamond,draw=WorkLFBlue,fill=white,line width=.8pt,inner sep=1.9pt]\n    at (.982894,.105409){};\n  \\node[diamond,draw=WorkLFTeal,fill=white,line width=.8pt,inner sep=1.9pt]\n    at (.105409,.982894){};\n\\end{scope}\n\\node[anchor=north west,align=left,text width=10.3cm,inner sep=0pt] at (4.55,9.18)\n  {Counts: $(3,0),(4,0),(3,1),(0,3),(0,4),(1,3)$.\\\\[3pt]\n   Regular IDF: $\\delta_a=\\delta_b=\\ln(1.5)$, then row L2 normalization.\\\\[3pt]\n   $\\mu_1=\\left(\\frac23+\\frac1{\\sqrt{10}},\\frac1{3\\sqrt{10}}\\right)\n     \\simeq(0.982894,0.105409)$. $\\mu_2$ reverses these coordinates.\\\\[3pt]\n   Passage 3 selects cluster 1 using its full vector:\\\\[2pt]\n   $R[3,a]=|0.948683-0.982894|\\simeq0.034211$.};\n\\node[anchor=west,font=\\fontsize{8.5}{10.5}\\selectfont] at (0,6.08)\n  {Circles / squares: passages in clusters 1 / 2. Diamonds: fitted centers.};\n\\draw[WorkLFRule,line width=.7pt] (0,5.78)--(15.05,5.78);\n\\node[title] at (0,5.43){(b) Keep every passage word record};\n\\node[title] at (9.05,5.43){(c) Read the endpoint columns};\n\\fill[WorkLFBlue!7] (0,4.91) rectangle (7.95,5.19);\n\\foreach \\x/\\lab in {.80/Passage,2.0/Word,3.15/Cluster,4.4/Weight,5.75/Center,7.15/Distance}\n  {\\node[font=\\fontsize{9}{10.5}\\selectfont\\bfseries] at (\\x,5.055){\\lab};}\n\\draw[WorkLFBlue,line width=.5pt] (0,4.89)--(7.95,4.89);\n\\foreach \\i/\\w/\\g/\\v/\\m/\\r/\\row in {\n  1/a/1/1.0000/0.9829/0.0171/0,\n  2/a/1/1.0000/0.9829/0.0171/1,\n  3/a/1/0.9487/0.9829/0.0342/2,\n  4/a/2/0.0000/0.1054/0.1054/3,\n  5/a/2/0.0000/0.1054/0.1054/4,\n  6/a/2/0.3162/0.1054/0.2108/5,\n  1/b/1/0.0000/0.1054/0.1054/6,\n  2/b/1/0.0000/0.1054/0.1054/7,\n  3/b/1/0.3162/0.1054/0.2108/8,\n  4/b/2/1.0000/0.9829/0.0171/9,\n  5/b/2/1.0000/0.9829/0.0171/10,\n  6/b/2/0.9487/0.9829/0.0342/11}{\n  \\pgfmathsetmacro{\\rowy}{4.65-.325*\\row}\n  \\node at (.80,\\rowy){\\i}; \\node at (2.0,\\rowy){$\\w$};\n  \\node at (3.15,\\rowy){\\g}; \\node at (4.4,\\rowy){\\v};\n  \\node at (5.75,\\rowy){\\m}; \\node[text=WorkLFBlue] at (7.15,\\rowy){\\r};\n}\n\\draw[WorkLFRule,line width=.4pt] (0,2.86)--(7.95,2.86);\n\\draw[WorkLFBlue,line width=.5pt] (0,.87)--(7.95,.87);\n\\node[anchor=west,font=\\fontsize{8.5}{10.5}\\selectfont] at (0,.54)\n  {Displayed entries are rounded. The CV uses full precision distances.};\n\\draw[->,WorkLFBlue,line width=.8pt] (8.12,3.03)--(8.83,3.03);\n\\node[word] (wa) at (10.05,4.35){Word $a$};\n\\node[cluster] (g1) at (13.8,4.35){Cluster 1};\n\\node[word] (wb) at (13.8,1.45){Word $b$};\n\\node[cluster] (g2) at (10.05,1.45){Cluster 2};\n\\foreach \\offset in {-.07,0,.07}{\n  \\draw[->,WorkLFBlue!75,line width=.55pt]\n    ([yshift=\\offset cm]wa.east)--([yshift=\\offset cm]g1.west);\n  \\draw[->,WorkLFTeal!85,line width=.55pt]\n    ([yshift=\\offset cm]wb.west)--([yshift=\\offset cm]g2.east);\n  \\draw[->,WorkLFTeal!85,line width=.55pt]\n    ([xshift=\\offset cm]wa.south)--([xshift=\\offset cm]g2.north);\n  \\draw[->,WorkLFBlue!75,line width=.55pt]\n    ([xshift=\\offset cm]wb.north)--([xshift=\\offset cm]g1.south);\n}\n\\node[anchor=south] at (11.93,4.58){Passages 1,2,3};\n\\node[anchor=north] at (11.93,1.22){Passages 4,5,6};\n\\node[rotate=90,anchor=south] at (9.78,2.9){Passages 4,5,6};\n\\node[rotate=90,anchor=north] at (14.06,2.9){Passages 1,2,3};\n\\node[align=center,text width=2.9cm,text=WorkLFBlue] at (11.93,3.05)\n  {$\\mathrm{CV}_a=\\mathrm{CV}_b$\\\\[2pt]$\\simeq0.842277$};\n\\node[align=center,text width=2.8cm] at (11.93,2.23){One node attribute\\\\per word};\n\\node[anchor=west,font=\\fontsize{8.5}{10.5}\\selectfont] at (9.05,.54)\n  {Twelve edges, one per long record.};\n\\node[anchor=west] at (0,-.04)\n  {Distance is separation, not affinity. Word CV is computed before the network is drawn.};\n\\end{tikzpicture}%\n\\endgroup%\n\n\\end{adjustbox}'
        render_diagram("Work_longformat", body)


## Feature fusion and supervised training

Define the explanatory figure using the shared drawing and export functions. The final rendering cell produces the illustration after computational verification.


In [ ]:
with report_failure('Feature fusion and supervised training', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def figure_headfusion():
        require_approval()
        body = '\n\\centering\n\\begin{adjustbox}{max width=\\linewidth}\n\n\\definecolor{FlowBlue}{HTML}{205F88}\n\\definecolor{FlowTeal}{HTML}{317D7B}\n\\begin{tikzpicture}[font=\\rmfamily\\small,>=Stealth,\n box/.style={draw=FlowBlue,rounded corners,align=center,text width=3.5cm,minimum height=1.05cm,inner sep=6pt},\n learn/.style={box,draw=FlowTeal,line width=1.2pt},\n flow/.style={->,line width=.8pt,draw=FlowBlue},\n back/.style={->,dashed,line width=1pt,draw=FlowTeal}]\n\\node[box] (text) at (0,0) {Ordered passage\\\\and target span};\n\\node[box] (corpus) at (5.2,0) {Training corpus\\\\word and cluster identities};\n\\node[learn] (bert) at (0,-2) {BERT: last block trainable\\\\target state $h_o\\in\\mathbb R^{768}$};\n\\node[box] (weight) at (5.2,-2) {Local TF-IDF importance\\\\$w_o\\in\\mathbb R$};\n\\node[box] (profile) at (5.2,-4) {Contextual behavior\\\\$\\beta_o\\in\\mathbb R^{2ks+2}$};\n\\node[box] (join) at (0,-4) {Same occurrence: concatenate\\\\$u_o\\in\\mathbb R^{771+2ks}$};\n\\node[learn] (head) at (0,-6) {Learned joint prediction\\\\$\\tanh(Au_o+b)$ then class scores};\n\\node[box] (labels) at (5.2,-6) {Independent occurrence labels\\\\$C$ task classes};\n\\node[learn] (loss) at (0,-8) {Cross-entropy loss\\\\updates head and last BERT block};\n\\draw[flow] (text)--(bert);\n\\draw[flow] (corpus)--(weight);\n\\draw[flow] (corpus.east)--++(.8,0)|-(profile.east);\n\\draw[flow] (bert)--(join);\n\\draw[flow] (weight.west)--++(-.5,0)|-(join.north east);\n\\draw[flow] (profile)--(join);\n\\draw[flow] (join)--(head);\n\\draw[flow] (head)--(loss);\n\\draw[flow] (labels.south)--(5.2,-7.05)--(2.5,-7.05)--(2.5,-8)--(loss.east);\n\\draw[back] (loss.west)--++(-1,0)|-(bert.west);\n\\node[align=left,text width=3.7cm] at (5.2,-8) {$k$: corpus-selected clusters\\\\$s=8$: declared statistics\\\\Solid: forward inputs\\\\Dashed: gradient path};\n\\end{tikzpicture}\n\n\\end{adjustbox}'
        render_diagram("Work_headfusion", body)


## Grouped population moments

Define the coordinate-distance aggregation needed to reconstruct plotted word measurements from the saved numerical reference.


In [ ]:
with report_failure('Grouped population moments', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def aggregate_distance_groups(distances, cluster_ids):
        """Exactly combine distance moments across groups, separately on each axis.

        Input measurements are distances only. Group means here are means of
        distances, not TF-IDF values or contextual-centroid coordinates. CV is
        formed from column-scaled centered moments, avoiding raw-square underflow
        and cancellation. Original-unit moments are retained as metadata, with
        representability masks. Scaled moments remain available when an original
        squared moment lies outside float64's range.
        """
        distances = np.asarray(distances, dtype=float)
        cluster_ids = np.asarray(cluster_ids)
        if distances.ndim != 2 or min(distances.shape, default=0) == 0 or cluster_ids.shape != (len(distances),):
            raise ValueError('Distances must have positive row and vocabulary widths, with one cluster identifier per row.')
        if cluster_ids.dtype.kind not in 'iu':
            raise ValueError('Cluster identifiers must be integers and retain their identities in group summaries.')
        if not np.isfinite(distances).all() or (distances < 0).any():
            raise ValueError('Coordinate distances must be finite and nonnegative.')
        scale = np.max(distances, axis=0)
        scaled = np.zeros_like(distances)
        np.divide(distances, scale, out=scaled, where=scale > 0)
        summaries=[]
        for g in np.unique(cluster_ids):
            block=scaled[cluster_ids==g]
            mean=block.mean(axis=0)
            total=block.sum(axis=0)
            square_total=(block**2).sum(axis=0)
            centered_total=((block-mean)**2).sum(axis=0)
            with np.errstate(over='ignore', under='ignore'):
                summaries.append(dict(cluster=int(g),count=len(block),sum=scale*total,
                    sum_squares=scale*(scale*square_total),mean=scale*mean,
                    centered_sum_squares=scale*(scale*centered_total),
                    scaled_sum=total,scaled_mean=mean,
                    scaled_sum_squares=square_total,
                    scaled_centered_sum_squares=centered_total))
        n=sum(group['count'] for group in summaries)
        scaled_mean=sum(group['scaled_sum'] for group in summaries)/n
        # Stable counterpart of Q/n-(S/n)^2, retaining between-group mean shifts.
        scaled_variance=sum(group['scaled_centered_sum_squares']+
            group['count']*(group['scaled_mean']-scaled_mean)**2
            for group in summaries)/n
        valid=scale > 0
        score=np.full_like(scaled_mean,np.nan)
        np.divide(np.sqrt(scaled_variance),scaled_mean,out=score,where=valid)
        with np.errstate(over='ignore', under='ignore'):
            mean=scale*scaled_mean
            standard_deviation=scale*np.sqrt(scaled_variance)
            variance=standard_deviation**2
        mean_representable=np.isfinite(mean) & ((mean > 0) | ~valid)
        variance_representable=np.isfinite(variance) & (
            (variance > 0) | (scaled_variance == 0))
        return dict(mean=mean,variance=variance,cv=score,valid=valid,groups=summaries,
            distance_scale=scale,scaled_mean=scaled_mean,scaled_variance=scaled_variance,
            mean_representable=mean_representable,
            variance_representable=variance_representable)

## Classifier names and figure interfaces

Associate the saved classifier identifiers with their reader-facing names and expose the moment calculation to the empirical plot functions.


In [ ]:
with report_failure('Classifier names and figure interfaces', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    from types import SimpleNamespace
    ORIGINAL_MOMENTS = SimpleNamespace(aggregate_distance_groups=aggregate_distance_groups)
    CLASSIFIERS = ["tfidf_logistic", "legacy_frozen_bert_only", "legacy_frozen_svd_cv", "bert_only", "bert_tfidf", "bert_profile", "bert_full", "bert_full_svd"]
    CLASSIFIER_LABELS = {
        "tfidf_logistic": "Term frequency--inverse document frequency logistic regression",
        "legacy_frozen_bert_only": "Frozen bidirectional transformer",
        "legacy_frozen_svd_cv": "Frozen bidirectional transformer with singular value decomposition and cluster identity",
        "bert_only": "Bidirectional transformer, final block trained",
        "bert_tfidf": "Bidirectional transformer with term frequency--inverse document frequency",
        "bert_profile": "Bidirectional transformer with contextual profile",
        "bert_full": "Bidirectional transformer with term frequency--inverse document frequency and contextual profile",
        "bert_full_svd": "Full feature combination with singular value decomposition",
    }


## Text escaping for TeX

Escape labels before passing them to TeX so data text does not become unintended typesetting syntax.


In [ ]:
with report_failure('Text escaping for TeX', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def tex(s):
        """Escape data-originated labels, leaving mathematical labels explicit."""
        mapping = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                   "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}",
                   "^": r"\textasciicircum{}"}
        return "".join(mapping.get(c, c) for c in str(s))

## Empirical figure export and provenance

Define how empirical plots, captions and source checksums are saved together for verification against the measured artifacts.


In [ ]:
with report_failure('Empirical figure export and provenance', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def save_figure(fig, output, stem, title, caption, prov):
        """Write vector PDF, rasterize that PDF for an exact 300 dpi preview."""
        import fitz
        from PIL import Image, PngImagePlugin
        output.mkdir(parents=True, exist_ok=True)
        pdf, png = output / (stem + ".pdf"), output / (stem + ".png")
        metadata = dict(prov)
        if pdf.exists():
            with fitz.open(pdf) as old:
                try:
                    before = json.loads(old.metadata.get("keywords", "{}"))
                    metadata["created_at_utc"] = before.get("created_at_utc", metadata["created_at_utc"])
                    metadata["created_by"] = before.get("created_by", metadata["created_by"])
                except (json.JSONDecodeError, AttributeError):
                    pass
        metadata.update(updated_at_utc=utc(), updated_by=prov["created_by"])
        with tempfile.TemporaryDirectory(prefix="work-figure-") as tmp:
            temporary_pdf = Path(tmp) / "render.pdf"
            fig.savefig(temporary_pdf, format="pdf", bbox_inches="tight", pad_inches=.08)
            with fitz.open(temporary_pdf) as doc:
                created = datetime.fromisoformat(metadata["created_at_utc"].replace("Z", "+00:00"))
                updated = datetime.fromisoformat(metadata["updated_at_utc"].replace("Z", "+00:00"))
                doc.set_metadata({
                    "title": title, "subject": caption, "creator": prov["created_by"],
                    "producer": "Matplotlib PGF / XeLaTeX / PyMuPDF",
                    "keywords": json.dumps(metadata, sort_keys=True),
                    "creationDate": created.strftime("D:%Y%m%d%H%M%SZ"),
                    "modDate": updated.strftime("D:%Y%m%d%H%M%SZ"),
                })
                if doc.page_count != 1:
                    raise AssertionError("A result figure must have one page.")
                image_count = len(doc[0].get_images(full=True))
                if image_count:
                    raise AssertionError("Expected a fully vector PDF, found embedded raster images.")
                fonts = sorted({font[3] for font in doc[0].get_fonts(full=True)})
                if not any("LMRoman" in f for f in fonts):
                    raise AssertionError(f"Latin Modern Roman was not embedded: {fonts}")
                doc.save(pdf)
                pix = doc[0].get_pixmap(matrix=fitz.Matrix(300 / 72, 300 / 72), alpha=False)
                image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                info = PngImagePlugin.PngInfo()
                info.add_text("Title", title)
                info.add_text("Description", caption)
                info.add_text("Provenance", json.dumps(metadata, sort_keys=True))
                image.save(png, pnginfo=info, dpi=(300, 300))
        return {
            "stem": stem, "title": title, "caption": caption, "provenance": metadata,
            "pdf": {"path": str(pdf.resolve()), "sha256": sha(pdf), "fonts": fonts, "raster_images": image_count},
            "png": {"path": str(png.resolve()), "sha256": sha(png), "dpi": 300},
        }

## Cluster score validation

Check that the saved curve contains every candidate count and that its selected count follows the maximum finite score and exact tie rule.


In [ ]:
with report_failure('Cluster score validation', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def validate_sweep(report):
        rows = report.get("results")
        if not isinstance(rows, list) or [r.get("k") for r in rows] != list(range(2, 101)):
            raise ValueError("The completed sweep must contain exactly one ordered row for every k=2..100.")
        finite = []
        for r in rows:
            if not r.get("status"):
                raise ValueError(f"Missing execution status for k={r['k']}.")
            value = r.get("CH")
            if value is not None:
                if not isinstance(value, (int, float)) or not math.isfinite(value) or value < 0:
                    raise ValueError(f"Invalid CH value for k={r['k']}; undefined scores must be null.")
                finite.append((r["k"], float(value)))
            for split in ("train", "validation", "test"):
                v = r.get(split + "_accuracy")
                if v is not None and (not math.isfinite(v) or not 0 <= v <= 1):
                    raise ValueError(f"Invalid {split} accuracy for k={r['k']}.")
        selected = report.get("selected_k_by_CH")
        if finite:
            winner, maximum = max(finite, key=lambda kv: (kv[1], -kv[0]))
            if selected != winner or not np.isclose(report.get("selected_CH", np.nan), maximum, rtol=1e-12, atol=0):
                raise ValueError("Stored selected k/CH disagrees with largest finite CH and smaller-k exact-tie rule.")
        elif report.get('cluster_count_selection_available', selected is not None):
            raise ValueError("A selected k was stored although every CH score is undefined.")
        return rows

## Agreement between cluster result formats

Check that the tabular cluster scores agree with the numerical report before plotting either representation.


In [ ]:
with report_failure('Agreement between cluster result formats', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def check_sweep_csv(path, rows):
        with path.open(newline="") as f:
            saved = list(csv.DictReader(f))
        if len(saved) != 99:
            raise ValueError("CSV and JSON sweep row counts differ.")
        for a, b in zip(rows, saved):
            if int(b["k"]) != a["k"] or b["status"] != a["status"]:
                raise ValueError("CSV and JSON sweep identity/status differ.")
            for field in ("CH", "train_accuracy", "validation_accuracy", "test_accuracy"):
                x, text = a.get(field), b.get(field, "")
                y = None if text in ("", "None", "null", "nan") else float(text)
                if (x is None) != (y is None) or (x is not None and not np.isclose(x, y, rtol=1e-12, atol=0)):
                    raise ValueError(f"CSV and JSON differ at k={a['k']}, field={field}.")

## Word-level measurement reconstruction

Reconstruct a word’s coordinate distances and grouped population moments from the stored reference for the word-level figure.


In [ ]:
with report_failure('Word-level measurement reconstruction', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def word_measurements(core, corpus, weights, ref, requested_word="space"):
        vocabulary = corpus["vocabulary"]
        if len(vocabulary) != weights.shape[1] or len(corpus["snippets"]) != weights.shape[0]:
            raise ValueError("Corpus vocabulary/snippet rows do not align with the weight matrix.")
        mask = np.asarray(ref["training_mask"], dtype=bool)
        if mask.shape != (weights.shape[0],) or not mask.any():
            raise ValueError("Training mask has invalid dimensions or is empty.")
        declared = np.array([s["split"] == "train" for s in corpus["snippets"]])
        if not np.array_equal(mask, declared):
            raise ValueError("Stored training mask differs from the corpus split assignment.")
        ids = np.asarray(ref["training_cluster_ids"])
        full_ids = np.asarray(ref["cluster_ids"])
        centers = np.asarray(ref["centroids"])
        if ids.shape != (int(mask.sum()),) or not np.array_equal(ids, full_ids[mask]):
            raise ValueError("Training cluster assignments do not align with the stored complete assignments.")
        if ids.dtype.kind not in "iu" or centers.ndim != 2 or centers.shape[1] != weights.shape[1]:
            raise ValueError("Invalid centroid/assignment dimensions or types.")
        if ids.min() < 0 or ids.max() >= len(centers):
            raise ValueError("An assignment is outside the fixed centroid array.")
        if requested_word in vocabulary:
            t = vocabulary.index(requested_word)
            selection = "Requested example word is present in the fitted training vocabulary."
        else:
            prevalence = np.asarray((weights[mask] > 0).sum(axis=0)).ravel()
            t = min(range(len(vocabulary)), key=lambda j: (-prevalence[j], vocabulary[j]))
            selection = "Requested word absent; chose highest training nonzero prevalence, lexicographic tie break."
        x = weights[mask, t].toarray().ravel()
        assigned = centers[ids, t]
        distances = np.abs(x - assigned)
        aggregated = core.aggregate_distance_groups(distances[:, None], ids)
        for key, expected in [("cv", ref["cv"][t]), ("mean", ref["cv_distance_mean"][t]),
                              ("variance", ref["cv_distance_variance"][t])]:
            if not np.isclose(aggregated[key][0], expected, rtol=2e-10, atol=1e-14, equal_nan=True):
                raise AssertionError(f"Reconstructed selected-word {key} disagrees with the fitted reference.")
        if bool(aggregated["valid"][0]) != bool(ref["cv_valid"][t]):
            raise AssertionError("Selected-word CV validity disagrees with the fitted reference.")
        n = len(x)
        groups = []
        for g in aggregated["groups"]:
            j = g["cluster"]
            if g["count"] != int(ref["cv_group_counts"][j]):
                raise AssertionError("A selected-word group count differs from the fitted reference.")
            if not np.isclose(g["mean"][0], ref["cv_group_mean"][j, t], rtol=2e-10, atol=1e-14):
                raise AssertionError("A selected-word group distance mean differs from the fitted reference.")
            if not np.isclose(g["centered_sum_squares"][0], ref["cv_group_centered_ss"][j, t], rtol=2e-10, atol=1e-14):
                raise AssertionError("A selected-word group centered sum differs from the fitted reference.")
            within = float(g["centered_sum_squares"][0] / n)
            between = float(g["count"] * (g["mean"][0] - aggregated["mean"][0]) ** 2 / n)
            groups.append(dict(cluster=j, count=g["count"], mean_distance=float(g["mean"][0]),
                               within_variance_contribution=within, between_mean_variance_contribution=between))
        reconstructed_variance = sum(g["within_variance_contribution"] + g["between_mean_variance_contribution"] for g in groups)
        if not np.isclose(reconstructed_variance, aggregated["variance"][0], rtol=2e-10, atol=1e-14):
            raise AssertionError("The displayed group components do not reconstruct the pooled distance variance.")
        order = np.lexsort((np.flatnonzero(mask), ids))
        return dict(word=vocabulary[t], column=t, selection=selection, values=x, centers=assigned,
                    distances=distances, ids=ids, snippet_rows=np.flatnonzero(mask), order=order,
                    aggregated=aggregated, groups=groups, n=n, k=len(centers))

## Underlying plot values

Define tabular export of the values used in each empirical figure so readers can reproduce the plotted measurements.


In [ ]:
with report_failure('Underlying plot values', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def write_csv(path, fieldnames, records, prov):
        path.parent.mkdir(parents=True, exist_ok=True)
        metadata = dict(prov)
        if path.exists():
            first = path.open().readline()
            if first.startswith("# provenance: "):
                before = json.loads(first.removeprefix("# provenance: "))
                metadata["created_at_utc"] = before.get("created_at_utc", metadata["created_at_utc"])
                metadata["created_by"] = before.get("created_by", metadata["created_by"])
        metadata.update(updated_at_utc=utc(), updated_by=prov["created_by"])
        with path.open("w", newline="") as f:
            f.write("# provenance: " + json.dumps(metadata, sort_keys=True) + "\n")
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(records)
        return {"path": str(path.resolve()), "sha256": sha(path), "provenance": metadata}

## Classification report validation

Validate completed classification records and their seed inventory before constructing comparative plots.


In [ ]:
with report_failure('Classification report validation', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def validate_classification_report(report, variant):
        if not report.get("complete"):
            raise ValueError(f"Classification result {variant} is incomplete; no final comparison will be plotted.")
        results = report.get("results", [])
        if not results or any(r.get("variant") != variant for r in results):
            raise ValueError(f"Classification results do not match the filename variant {variant}.")
        seeds = [r.get("seed") for r in results]
        expected = [29] if variant == "tfidf_logistic" else report.get("configuration", {}).get("seeds")
        if len(set(seeds)) != len(seeds) or expected is None or set(seeds) != set(expected):
            raise ValueError(f"Missing or duplicate executed seed results for {variant}.")
        if variant != "tfidf_logistic" and set(seeds) != {19, 29, 41}:
            raise ValueError(f"Expected the declared three-seed comparison for {variant}, found {seeds}.")
        for r in results:
            for key in ("accuracy", "macro_f1"):
                x = r.get("test", {}).get(key)
                if x is None or not math.isfinite(x) or not 0 <= x <= 1:
                    raise ValueError(f"Invalid held-out {key} for {variant}.")
        return sorted(results, key=lambda r: r["seed"])

## Variance ratio criterion across cluster counts

Define the plot of all 99 candidate counts, marking the selected count and exact criterion value independently of classification accuracy.


In [ ]:
with report_failure('Variance ratio criterion across cluster counts', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def plot_sweep(plt, folder, output, executor):
        require_approval()
        require_verified_results(folder)
        path=folder/'cluster_sweep.json'
        report=json.loads(path.read_text()); rows=validate_sweep(report)
        csv_path=folder/'cluster_sweep_scores.csv'; check_sweep_csv(csv_path,rows)
        k=np.array([r['k'] for r in rows]); ratio=np.array([np.nan if r['CH'] is None else r['CH'] for r in rows])
        fig,ax=plt.subplots(figsize=(10,4.5))
        ax.plot(k,ratio,color=BLUE,marker='.',markersize=3)
        ax.set_xticks(k); ax.tick_params(axis='x',labelrotation=90,labelsize=5)
        ax.set(xlabel='Number of clusters',ylabel='Variance ratio criterion (Caliński–Harabasz index)')
        selected=report.get('selected_k_by_CH') if report.get('cluster_count_selection_available',True) else None
        if selected is not None:
            score=report['selected_CH']; ax.scatter([selected],[score],color=TEAL,zorder=4)
            ax.set_title(f'Selected k={selected}; variance ratio criterion={score:.17g}')
        else: ax.set_title('No finite variance ratio criterion in the declared search range')
        fig.tight_layout()
        caption='Every tested cluster count from2 through100 is plotted. The maximum finite between-to-within mean-square ratio selects the count, with the smaller count preferred on an exact tie. Classification results are shown in a separate figure.'
        prov=provenance('Plot the verified variance ratio criterion for every declared cluster count',[path,csv_path],executor)
        result=save_figure(fig,output,'Work_cluster_selection','Variance ratio criterion across cluster counts',caption,prov)
        plt.close(fig)
        result.update(selected_k=selected,selected_CH=report.get('selected_CH'),plotted_k=k.tolist())
        return result


## Word distances and grouped moments

Define the empirical word figure using the saved partitions, chunks, vocabulary and numerical reference, with its underlying measurements exported alongside it.


In [ ]:
with report_failure('Word distances and grouped moments', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def plot_word(plt, core, folder, output, executor):
        require_approval()
        require_verified_results(folder)
        sources = [folder / f for f in ("source_partitions.json", "weights.npz", "reference.npz", "vocabulary.json")]
        partitions=json.loads(sources[0].read_text())
        corpus=dict(documents=partitions['documents'],snippets=[],vocabulary=json.loads(sources[3].read_text())['words'])
        for document in partitions['documents']:
            ends=json.loads((folder/'segments'/(document['document_id']+'.json')).read_text())['ends']
            corpus['snippets'].extend(dict(doc=document['id'],split=document['split']) for end in ends)
        weights = sparse.load_npz(sources[1]).tocsr()
        with np.load(sources[2]) as ref:
            m = word_measurements(core, corpus, weights, ref)
        prov = provenance("Plot one actual word using unchanged original grouped-distance aggregation", sources, executor)
        n, order, word, groups = m["n"], m["order"], m["word"], m["groups"]
        positions = np.arange(1, n + 1)
        fig, axes = plt.subplots(3, 1, figsize=(6.5, 6.5), gridspec_kw={"height_ratios": [1, .8, 1.1]})
        fig.subplots_adjust(left=.15, right=.98, bottom=.13, top=.94, hspace=.52)
        axes[0].plot(positions, m["values"][order], linestyle="none", marker=".", markersize=1.8,
                     alpha=.6, color=BLUE, label="Normalized TF--IDF weight")
        axes[0].plot(positions, m["centers"][order], color=TEAL, linewidth=1.1,
                     label="Assigned centroid's same word coordinate")
        axes[0].set_title("(a) Word ``" + tex(word) + "'': all training snippets, grouped by fixed assignment", loc="left")
        axes[0].set_ylabel("Weight / coordinate")
        axes[0].legend(loc="upper right", fontsize=8.5)
        axes[1].plot(positions, m["distances"][order], linestyle="none", marker=".",
                     markersize=1.8, alpha=.6, color=BLUE)
        axes[1].set_title(r"(b) Coordinate distance $r_i=|x_{it}-c_{g(i)t}|$", loc="left")
        axes[1].set_ylabel("Absolute distance")
        axes[1].set_xlabel("Training snippet position after sorting by fixed cluster assignment")
        edges = np.cumsum([g["count"] for g in groups])
        for ax in axes[:2]:
            ax.set_xlim(.5, n + .5)
            ax.grid(axis="y")
            for edge in edges[:-1]:
                ax.axvline(edge + .5, color=PALE, linewidth=.45)
        axes[0].tick_params(labelbottom=False)
        cluster = np.array([g["cluster"] for g in groups])
        within = np.array([g["within_variance_contribution"] for g in groups])
        between = np.array([g["between_mean_variance_contribution"] for g in groups])
        axes[2].bar(cluster, within, color=BLUE, width=.75, label=r"Within: $SS_g/n$")
        axes[2].bar(cluster, between, bottom=within, color=TEAL, edgecolor="white", linewidth=.2,
                    hatch="//", width=.75, label=r"Between means: $n_g(\bar r_g-\bar r)^2/n$")
        axes[2].set_title("(c) Count-weighted contributions to pooled population variance", loc="left")
        axes[2].set_xlabel("Fixed cluster identifier (as stored)")
        axes[2].set_ylabel("Variance contribution")
        axes[2].grid(axis="y")
        if len(cluster) <= 15:
            axes[2].set_xticks(cluster)
        axes[2].legend(loc="upper right", fontsize=8.5)
        avg, variance, cv = (float(m["aggregated"][key][0]) for key in ("mean", "variance", "cv"))
        cv_label = f"{cv:.17g}" if np.isfinite(cv) else "undefined (zero mean distance)"
        fig.text(.15, .025, f"{n:,} training snippets; {m['k']} fixed clusters; zero word weights included.\n"
                 + r"Pooled $\bar r=$" + f"{avg:.10g}; SD = {np.sqrt(variance):.10g}; CV = {cv_label}.", fontsize=9)
        caption = (
            f"Observed word '{word}' (vocabulary column {m['column']}) in all {n} training snippets. "
            "Snippets are ordered by their stored full-vector cluster assignments, with original row order "
            "retained within each cluster. Weight and centroid-coordinate values are dimensionless; all "
            "zero word weights are included. Each distance is the absolute difference from the matching "
            "coordinate of that snippet's assigned centroid. Bars show each cluster's centered distance "
            "sum divided by total n, plus its count-weighted distance-mean shift divided by n. Their sum "
            "is the pooled population variance of distances, not the variance of TF-IDF weights. "
            "The original Paper One aggregation recomputes and verifies the stored CV."
        )
        result = save_figure(fig, output, "Work_word_aggregation", "Actual word-coordinate distance aggregation", caption, prov)
        plt.close(fig)
        measurements = []
        for rank, i in enumerate(order, 1):
            row = int(m["snippet_rows"][i])
            snippet = corpus["snippets"][row]
            doc = corpus["documents"][snippet["doc"]]
            measurements.append(dict(word=word, vocabulary_column=m["column"], plot_position=rank,
                snippet_row=row, source_document_id=doc.get("document_id", doc.get("id")),
                cluster=int(m["ids"][i]), normalized_tfidf_weight=float(m["values"][i]),
                assigned_centroid_coordinate=float(m["centers"][i]), absolute_coordinate_distance=float(m["distances"][i])))
        result["measurements"] = write_csv(output / "Work_selected_word_measurements.csv", list(measurements[0]), measurements, prov)
        result["group_components"] = write_csv(output / "Work_selected_word_group_components.csv", list(groups[0]), groups, prov)
        result.update(word=word, word_selection=m["selection"], training_snippets=n, clusters=m["k"],
                      pooled_distance_mean=avg, pooled_distance_variance=variance,
                      pooled_distance_CV=cv if np.isfinite(cv) else None, original_aggregation_verified=True)
        return result

## Classification results on shared sources

Define the classification comparison plot and verify that every classifier refers to the same held-out document identifiers and labels.


In [ ]:
with report_failure('Classification results on shared sources', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    def plot_classification(plt, folder, output, executor, require_all=True):
        require_approval()
        require_verified_results(folder)
        missing = [v for v in CLASSIFIERS if not (folder / ("part_" + v + ".json")).exists()]
        if missing:
            if require_all:
                raise FileNotFoundError("Final classification comparison awaits: " + ", ".join(missing))
            return {"status": "NOT_READY", "missing_variants": missing}
        sources = [folder / ("part_" + v + ".json") for v in CLASSIFIERS]
        reports = [json.loads(p.read_text()) for p in sources]
        incomplete = [v for v, r in zip(CLASSIFIERS, reports) if not r.get("complete")]
        if incomplete and not require_all:
            return {"status": "NOT_READY", "incomplete_variants": incomplete}
        rows = [validate_classification_report(r, v) for r, v in zip(reports, CLASSIFIERS)]
        # Comparisons must refer to exactly the same held-out source IDs and labels.
        population = None
        for report in reports:
            predictions = report.get("predictions", {})
            if not predictions:
                raise ValueError("Cannot verify comparison population without stored source-level predictions.")
            for p in predictions.values():
                ids, labels = p["document_ids"], p["labels"]
                if len(ids) != len(labels) or len(set(ids)) != len(ids):
                    raise ValueError("Duplicated IDs or mismatched source labels in stored predictions.")
                signature = dict(zip(ids, labels))
                if population is None:
                    population = signature
                elif signature != population:
                    raise ValueError("Classifier variants do not share an identical held-out source population.")
        fig, axes = plt.subplots(1, 2, figsize=(6.5, 4.25), sharey=True)
        fig.subplots_adjust(left=.43, right=.98, bottom=.19, top=.89, wspace=.13)
        summaries = []
        y = np.arange(len(CLASSIFIERS))
        for ax, key, title in zip(axes, ("accuracy", "macro_f1"), ("Test accuracy", "Test macro F1")):
            for i, (variant, records) in enumerate(zip(CLASSIFIERS, rows)):
                values = np.array([r["test"][key] for r in records])
                avg = float(values.mean())
                sd = float(values.std(ddof=1)) if len(values) > 1 else None
                if sd is not None:
                    ax.errorbar(avg, i, xerr=sd, color=BLUE, marker="D", markersize=4,
                                linewidth=1.3, capsize=3, zorder=2)
                    for offset, r, value in zip((-.17, 0, .17), records, values):
                        shape = {19: "o", 29: "s", 41: "^"}[r["seed"]]
                        ax.plot(value, i + offset, marker=shape, markerfacecolor="white",
                                markeredgecolor=TEAL, markersize=3.6, linestyle="none", zorder=3)
                else:
                    ax.plot(avg, i, marker="D", color=GRAY, markersize=4, linestyle="none")
                summaries.append(dict(variant=variant, metric=key, number_of_executed_fits=len(values),
                                      mean=avg, sample_standard_deviation=sd,
                                      **{"seed_" + str(r["seed"]): float(v) for r, v in zip(records, values)}))
            ax.set_title(title)
            ax.set_xlim(0, 1)
            ax.set_xticks([0, .25, .5, .75, 1])
            ax.set_xticklabels(["0", ".25", ".5", ".75", "1"])
            ax.grid(axis="x")
            ax.set_ylim(len(CLASSIFIERS) - .5, -.5)
            ax.set_yticks(y)
        axes[0].set_yticklabels([CLASSIFIER_LABELS[v] for v in CLASSIFIERS], fontsize=8.5)
        axes[1].tick_params(labelleft=False)
        fig.text(.04, .065, "Open markers: seeds 19 (circle), 29 (square), 41 (triangle).\n"
                 "Diamonds: descriptive mean; bars: sample SD across three fits.\n"
                 "TF--IDF logistic has one fit; SD is unavailable. Bars are not confidence intervals.", fontsize=8.5)
        caption = (
            "Held-out source-document topic classification on an identical source population for every variant. "
            "Two axes show accuracy and macro F1 on their original 0-to-1 scales. The three open markers show "
            "seeds 19, 29 and 41; each diamond is their arithmetic mean and each horizontal bar is one sample "
            "standard deviation with denominator 2. Profile features include stored word statistics, their "
            "validity masks, between-cluster variance and the current occurrence's coordinate distance; "
            "they do not directly include pooled CV. SVD denotes raw and CV-weighted SVD features. The "
            "frozen SVD variant also includes a learned cluster embedding. The error bars show descriptive "
            "seed variability, not confidence "
            "intervals or population uncertainty. The TF-IDF logistic baseline is a single executed fit. "
            "All plotted variant files must declare completion; their held-out document IDs and labels are verified equal."
        )
        prov = provenance("Plot completed classification comparisons on matching held-out source documents", sources, executor)
        result = save_figure(fig, output, "Work_classification_comparison", "Executed classification comparison", caption, prov)
        plt.close(fig)
        fields = ["variant", "metric", "number_of_executed_fits", "mean", "sample_standard_deviation", "seed_19", "seed_29", "seed_41"]
        result["measurements"] = write_csv(output / "Work_classification_plot_values.csv", fields, summaries, prov)
        result.update(variants=CLASSIFIERS, held_out_source_documents=len(population), uncertainty="Descriptive sample SD across three seeds; not a confidence interval")
        return result

## Render the verified figures

After all generators are defined, require the completed computational verification record, render the figures and save the figure manifest.


In [ ]:
with report_failure('Render the verified figures', 'Check the explicit input paths, measured artifact completeness, array dimensions, output permissions and installed rendering dependencies. Do not substitute invented measurements.'):
    SYNTHETIC_FIGURE_FUNCTIONS = [figure_bridges, figure_topology, figure_semantic, figure_coordinates, figure_distance, figure_variance, figure_tree, figure_projections, figure_roles, figure_prevalence, figure_variance_three_dimensions, figure_ksearch, figure_longformat, figure_headfusion]

    if RENDER_FIGURES:
        require_approval()
        plt = configure_plotting()
        for generate in SYNTHETIC_FIGURE_FUNCTIONS:
            generate()
        empirical = []
        if RESULTS_DIRECTORY is not None:
            folder = Path(RESULTS_DIRECTORY)
            require_verified_results(folder)
            empirical = [plot_sweep(plt, folder, OUT, AUTHOR),
                         plot_word(plt, ORIGINAL_MOMENTS, folder, OUT, AUTHOR),
                         plot_classification(plt, folder, OUT, AUTHOR)]
        (OUT / "figure_manifest.json").write_text(json.dumps({"synthetic":values,"empirical":empirical}, indent=2, allow_nan=False))


## CBS occupation-classification figures

Independent renderer for the verified outputs of project_notebooks/cbs_occupation_classification. This section does not obtain datasets or run model training. The function reads explicit result paths and creates four figures. It is self-contained and can be run without executing the preceding research-paper sections.


In [ ]:
def render_cbs_assignment_figures(input_dir, output_dir):
    """Plot measured assignment outputs only. No dataset download or model training."""
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix
    source=Path(input_dir)
    destination=Path(output_dir)
    destination.mkdir(parents=True,exist_ok=True)
    run=json.loads((source/'run_results.json').read_text())
    predictions=pd.read_csv(source/'heldout_predictions.csv',dtype={'observed_code':str,'predicted_code':str})
    comparison=pd.read_csv(source/'model_comparison.csv')
    if len(predictions)!=191 or not all(run['checks'].values()):
        raise ValueError('Verified 191-row held-out results are required.')
    paths=[]
    with plt.rc_context({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False}):
        fig,ax=plt.subplots(figsize=(9,5.2),layout='constrained')
        grouped=comparison[(comparison['split']=='grouped') & (comparison['fields']=='context_and_core')]
        names=['Most frequent','Words','Characters','Words + characters']
        x=np.arange(len(grouped)); a=grouped.accuracy.to_numpy()*100
        bars=ax.bar(x,a,color=['#9AA7B1','#477B9F','#2D607F','#163F5E'])
        ax.set(xticks=x,xticklabels=names,ylim=(0,100),ylabel='Exact code agreement (%)',title='Grouped evaluation on 191 fully labelled records')
        ax.bar_label(bars,labels=[f'{v:.1f}%' for v in a],padding=4)
        ax.text(.02,.93,'Same five folds. No identical task-description group crosses folds.',transform=ax.transAxes,fontsize=10)
        p=destination/'model_comparison.png';fig.savefig(p,dpi=180);plt.close(fig);paths.append(p)

        fig,ax=plt.subplots(figsize=(8.8,5.2),layout='constrained')
        selected=comparison[(comparison.model=='combined') & (comparison['fields']=='context_and_core')]
        a=selected.accuracy.to_numpy()*100
        bars=ax.bar(['Grouped descriptions','Random rows'],a,color=['#163F5E','#A1ABB2'],width=.55)
        ax.set(ylim=(0,100),ylabel='Exact code agreement (%)',title='Evaluation changes when matching descriptions can cross folds')
        ax.bar_label(bars,labels=[f'{v:.1f}%' for v in a],padding=5)
        ax.text(.02,.92,'Sensitivity comparison, not production accuracy.',transform=ax.transAxes,fontsize=10)
        p=destination/'split_sensitivity.png';fig.savefig(p,dpi=180,bbox_inches='tight');plt.close(fig);paths.append(p)

        fig,ax=plt.subplots(figsize=(9,5.5),layout='constrained')
        counts=predictions.observed_code.value_counts()
        frequencies=counts.value_counts().sort_index()
        bars=ax.bar(frequencies.index.astype(str),frequencies.to_numpy(),color='#2D607F')
        ax.set(xlabel='Records per occupation code',ylabel='Number of codes',title='59 complete-label classes, including 24 singletons')
        ax.bar_label(bars,padding=3)
        p=destination/'class_support.png';fig.savefig(p,dpi=180);plt.close(fig);paths.append(p)

        fig,ax=plt.subplots(figsize=(9,5.5),layout='constrained')
        mapping={'ShemAvoda':'Workplace name','SugAvoda':'Workplace activity','ShemMachlaka':'Department name',
                 'SugMachlaka':'Department activity','EzoAvoda':'Work performed','TeurPeula':'Main activities','TeurTafkid':'Job title'}
        labels=list(mapping.values());values=[run['quality']['missing_by_field'][k] for k in mapping]
        bars=ax.barh(labels,values,color='#477B9F')
        ax.set(xlim=(0,200),xlabel='Missing values among 200 records',title='Text-field coverage in the supplied workbook')
        ax.bar_label(bars,padding=4);ax.invert_yaxis()
        p=destination/'missing_text.png';fig.savefig(p,dpi=180);plt.close(fig);paths.append(p)
    print('Rendered four figures from saved measured results.')
    return paths


## Render the CBS assignment figures

Set RENDER_CBS_ASSIGNMENT to True after running the assignment notebook. The paths below are relative to the repository root. This default leaves existing rendering behavior unchanged.


In [ ]:
RENDER_CBS_ASSIGNMENT = False
if RENDER_CBS_ASSIGNMENT:
    render_cbs_assignment_figures("project_notebooks/cbs_occupation_classification/results", "project_notebooks/cbs_occupation_classification/results/figures")
